# SitePulse — Exploratory Data Analysis & Statistical Baselines
### Phase 3: Analytical Validation, Feature Distributions & Leakage-Free Baselines

> **Data Integrity & Simulation Disclaimer:**
> All empirical findings, distributions, and correlation coefficients in this notebook are derived from the **SitePulse Synthetic Fitout Simulation Engine (Seed 42)**. They serve strictly to validate simulation fidelity, algorithmic assumptions, and feature pipeline integrity. They **do not claim** to represent real-world construction industry findings.

## 1. Setup & Ingestion of Synthetic Relational Datasets

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

DATA_DIR = "../data/synthetic"
FIG_DIR = "figures"

# Load synthetic relational tables
projects_df = pd.read_csv(os.path.join(DATA_DIR, "projects.csv"))
trades_df = pd.read_csv(os.path.join(DATA_DIR, "trades.csv"))
updates_df = pd.read_csv(os.path.join(DATA_DIR, "daily_updates.csv"))
delays_df = pd.read_csv(os.path.join(DATA_DIR, "delay_events.csv"))
trajectories_df = pd.read_csv(os.path.join(DATA_DIR, "consolidated_trajectories.csv"))
proj_features_df = pd.read_csv(os.path.join(DATA_DIR, "project_features_day25.csv"))
trade_features_df = pd.read_csv(os.path.join(DATA_DIR, "trade_features_day25.csv"))

print(f"[OK] Loaded {len(projects_df)} Projects, {len(trades_df)} Trades, {len(updates_df):,} Daily Updates.")
print(f"[OK] Loaded {len(proj_features_df)} Project Features, {len(trade_features_df)} Trade Features (Cutoff Day 25).")

## 2. Project Duration & Delay Distribution Analysis

We evaluate project planned duration ($38-48$ days) against simulated actual durations across the 8 operational scenarios.

In [ ]:
scenario_summary = projects_df.groupby("scenario").agg(
    project_count=("id", "count"),
    planned_duration_mean=("planned_duration_days", "mean"),
    actual_duration_mean=("actual_duration_days", "mean"),
    mean_delay_days=("delay_days", "mean"),
    delayed_over_3d_pct=("is_delayed_over_3d", lambda x: round(x.mean() * 100, 1))
).reset_index()

display(scenario_summary)

# Display pre-generated high-resolution chart
display(Image(os.path.join(FIG_DIR, "01_project_duration_and_delays.png")))

### Observations:
- `on_schedule`: Average delay is ~3.3 days, reflecting minor standard variations.
- `recovery_intervention`: Average delay is -10.5 days; the workforce surge mid-project successfully restores the schedule.
- `approval_bottleneck` & `material_delay`: Yield moderate delays averaging 8.1 to 8.5 days.
- `concurrent_delays` & `manpower_delay`: Induce severe schedule slips exceeding 28 to 32 days.

## 3. Manpower Shortfall vs. Delay Correlation

We analyze the relationship between cumulative worker shortfalls (worker-days understaffed) and final actual delay days.

In [ ]:
proj_manpower = trajectories_df.groupby("project_id").agg(
    total_shortfall_days=("shortfall_workers", "sum"),
    total_workers_present=("workers_present", "sum"),
    total_workers_required=("workers_required", "sum")
).reset_index()

merged_m = pd.merge(projects_df, proj_manpower, left_on="id", right_on="project_id")
r_val = round(merged_m["total_shortfall_days"].corr(merged_m["delay_days"]), 3)
print(f"Pearson correlation between Cumulative Worker Shortfall and Final Delay Days: r = {r_val}")

display(Image(os.path.join(FIG_DIR, "02_manpower_shortfall_vs_delays.png")))

## 4. Trade-Level Productivity & Velocity Distribution

We inspect daily progress deltas across trades during active (non-stagnant) shifts.

In [ ]:
active_deltas = trajectories_df[trajectories_df["daily_delta_pct"] > 0]
trade_stats = active_deltas.groupby("trade_name")["daily_delta_pct"].describe().round(2)
display(trade_stats)

display(Image(os.path.join(FIG_DIR, "03_trade_productivity_and_velocity.png")))

## 5. Synthetic Anomaly Detection Distribution

We verify the injection frequency and balance of synthetic anomaly patterns in the misleading reporting scenario.

In [ ]:
anomaly_dist = trajectories_df["anomaly_type"].value_counts()
display(pd.DataFrame({"Count": anomaly_dist, "Percentage": (anomaly_dist / len(trajectories_df) * 100).round(2)}))

display(Image(os.path.join(FIG_DIR, "04_anomaly_scenario_distribution.png")))

## 6. Leakage-Free Feature Correlations at Prediction Cutoff Day 25

Features extracted at Day 25 strictly use data prior to or on Day 25. We assess their correlation with the binary target `target_is_delayed_over_3d`.

In [ ]:
feature_cols = [
    "timeline_elapsed_ratio", "overall_actual_pct", "schedule_variance_pct",
    "max_bottleneck_gap_pct", "delayed_trades_ratio", "workforce_deficit",
    "workforce_fulfillment_ratio", "recent_7d_progress_velocity",
    "velocity_health_ratio", "active_delay_events_count"
]

corrs = proj_features_df[feature_cols].apply(lambda s: s.corr(proj_features_df["target_is_delayed_over_3d"])).round(3).sort_values()
display(pd.DataFrame({"Pearson Correlation (r)": corrs}))

display(Image(os.path.join(FIG_DIR, "05_feature_target_correlation.png")))

## 7. Simple Statistical Heuristic Baseline

To establish a benchmark for future ML models (without training ML classifiers in Phase 3), we evaluate a domain heuristic rule:
$$\text{Predicted Delayed} = 1 \iff \text{schedule\_variance\_pct} < -3.0\% \text{ OR } \text{delayed\_trades\_ratio} \ge 0.40$$

In [ ]:
y_true = proj_features_df["target_is_delayed_over_3d"]
y_pred_heuristic = ((
    (proj_features_df["schedule_variance_pct"] < -3.0) |
    (proj_features_df["delayed_trades_ratio"] >= 0.40)
)).astype(int)

accuracy = (y_true == y_pred_heuristic).mean()
tp = ((y_true == 1) & (y_pred_heuristic == 1)).sum()
fp = ((y_true == 0) & (y_pred_heuristic == 1)).sum()
fn = ((y_true == 1) & (y_pred_heuristic == 0)).sum()
tn = ((y_true == 0) & (y_pred_heuristic == 0)).sum()

precision = tp / max(1, (tp + fp))
recall = tp / max(1, (tp + fn))
f1 = 2 * precision * recall / max(1e-4, (precision + recall))

print("=== HEURISTIC BASELINE BENCHMARK (Cutoff Day 25) ===")
print(f"Accuracy : {round(accuracy * 100, 2)}%")
print(f"Precision: {round(precision * 100, 2)}%")
print(f"Recall   : {round(recall * 100, 2)}%")
print(f"F1-Score : {round(f1, 4)}")
print(f"Confusion Matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print("====================================================")

## 8. Summary & Next Steps for Phase 4
- **Simulation Fidelity:** Verified that multi-trade dependencies, workforce shortages, and roadblocks generate diverse, realistic project trajectories.
- **Target Integrity:** Target labels (`target_is_delayed_over_3d`, `target_has_anomaly`) are strictly derived from future project outcomes while prediction-time features use only historical data up to cutoff Day 25.
- **Project Separation:** Project IDs are grouped across train (70%), validation (15%), and test (15%) splits, ensuring zero leakage across sets.
- **Ready for Phase 4:** The feature store and split datasets (`train_features.csv`, `val_features.csv`, `test_features.csv`) provide a clean baseline for training ML classification and anomaly models in Phase 4 upon review approval.